# Taking derivatives on your computer

There are a number of ways that you can take computers on your derivative.

Today we'll discuss three:

* Numerical differentiation (approximation)
* Symbolic differentiation (exact)
* Automatic differentiation (exact)

As discussed in the lecture, we will have a strong preference for **automatic differentiation** anywhere that we can use.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

We will start by defining an example function that we can use:

$$f(x) = -\exp^{x_0^2 + x_1^2}$$

which gives us

$$\nabla f(x) = \begin{bmatrix} -2 x_0 f(x)) \\ -2 x_1 f(x) \end{bmatrix}$$

In [ ]:
def f(x):
    return -np.exp(-(x[0]**2 + x[1]**2))

def df(x):
    return -2*np.asarray(x)*f(x)

In [ ]:
f(np.array([0.0, 0.0]))

In [ ]:
df(np.array([0.0, 0.0]))

We will also bring the gradient descent algorithm that we created so we can experiment with the different solutions

In [ ]:
def grad_descent(df, x0, epsilon=1e-3, T=200, alpha=0.1):
    """
    Given a gradient function df, staritng starting point x0,
    stopping parameters epsilon and T, and a learning rate alpha;
    find a local minimum of f(x) near x_0 via gradient descent
    """
    x = np.copy(x0)
    trace = []
    for i in range(T):
        df_i = df(x)
        xp = x - alpha * df_i
        err = max(abs(df_i))
        status = {"x": xp, "i": i, "err": err}
        trace.append(status)
        if err < epsilon:
            return trace
        x[:] = xp[:]

    return trace

### Numerical differentiation

- Our example is an easily differentiable function...
- However, that won't always be the case and we want to be able to differentiate any function that we come across even if we don't know the derivative (or have time to write it out)
- In these cases, the standard approach is to approximate the derivative numerically
- The classic algorithm for numerical approximation of derivatives is called finite differencing

#### Finite Differences

- A finite difference approximation of a derivative comes directly from the definition of a derivative: $$\frac{df}{dx} = \lim_{\delta \downarrow 0} \frac{f(x + \delta) - f(x)}{\delta}$$
- In code, we can choose a value for $\delta$ that is very small -- perhaps on the order of 1e-6 -- and evaluate the fraction above
- For the gradient, we apply the finite difference approximation one element of $x$ at a time: $$\nabla f(x) \approx \begin{bmatrix}\frac{f(x + e_1 \delta) - f(x)}{\delta} \\ \frac{f(x + e_2 \delta) - f(x)}{\delta} \\ \vdots \\ \frac{f(x + e_N \delta) - f(x)}{\delta} \end{bmatrix},$$
where $e_i$ is the $i$th unit vector

In [ ]:
def forward_difference(f, x, delta):
    out = np.zeros_like(x)
    fx = f(x)
    
    for i in range(len(x)):
        xi = np.copy(x)
        xi[i] += delta
        fx_i = f(xi)
        out[i] = (fx_i - fx) / delta
    
    return out

In [ ]:
x0 = np.array([0.2, 0.4])
forward_difference(f, x0, 1e-5)

In [ ]:
df(x0)

In [ ]:
def grad_descent_finite_diff(f, x0, delta=1e-4, **kw):
    def df_fd(x):
        return forward_difference(f, x, delta=delta)
    return grad_descent(df_fd, x0, **kw)

- Let's see what happens in our example when we use numerical derivatives

In [ ]:
trace_fd = grad_descent_finite_diff(f, [2, -0.3])
trace_fd[-1]

#### Choosing $\delta$

- To use finite differencing techniques we need to choose a value for the parameter $\delta$
- In the mathematical theory, $\delta$ should approach zero to compute the exact derivative (remember the limit definition!!)
- However, computers don't do exact artithmetic
- Instead, they use floating poing approximations
- One implication of this is that dividing by a very small number (like a small $\delta$) can be highly inaccurate
- Let's explore this...

In [ ]:
def plot_fd_err(f, df, x0):
    x = []
    y = []
    dfdx = df(x0)
    for delta in np.logspace(-15, 0, 70):
        approx_dfdx = forward_difference(f, x0, delta=delta)
        x.append(delta)
        y.append(max(abs(dfdx - approx_dfdx)))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.loglog(x, y)
    ax.set_xlabel("delta")
    ax.set_ylabel("abs error in ∇f")
    return ax

plot_fd_err(f, df, [0.5, -0.5]);

#### Aside: Smoothness

- The function `f` we have been working with is particularly well behaved
- Many objective functions in machine learning are not
- When we have a less smooth function, choosing $\delta$ is even more difficult and important

#### Comment on Efficiency

- Note that in order to do a finite difference approximation of the gradient we needed N+1 function calls, where $N$ is the number of elements in $x$
- This is ok when $N = 2$
- But when we do deep learning, $N$ is the number of parameters in our model, which can be in the billions!!!
- Finite differences is too costly from a time perspective -- We would need to evaluate the 

## Symbolic differentiation

The next approach is **Symbolic differentiation**. Unlike numerical approximation, symbolic differentiation uses the rules of calculus (like the power rule, product rule, and chain rule) to find the exact algebraic expression for the derivative.

In Python, the most popular library for this is `sympy`. It allows us to treat variables as symbols and perform algebraic manipulations.

**Advantages:**
- It provides an **exact** solution (no $\delta$ approximation error).
- It can handle complex expressions and simplify them for us.

**Disadvantages:**
- It can suffer from "expression swell" where the derivative becomes computationally expensive to evaluate for very complex functions.
- It doesn't work with "black box" code (like a function containing a `for` loop or `if` statement).

In [ ]:
import sympy

# Define our symbols
x0, x1 = sympy.symbols('x0 x1')

# Define the function symbolically (using sympy's exp)
f_sym = -sympy.exp(-(x0**2 + x1**2))

# Display the function
f_sym

In [ ]:
# Compute partial derivatives
df_dx0 = sympy.diff(f_sym, x0)
df_dx1 = sympy.diff(f_sym, x1)

# Create the symbolic gradient vector
grad_f_sym = sympy.Matrix([df_dx0, df_dx1])

print("Symbolic Gradient:")
grad_f_sym

In [ ]:
f_num = sympy.lambdify((x0, x1), f_sym, 'numpy')
df_num_raw = sympy.lambdify((x0, x1), [df_dx0, df_dx1], 'numpy')


def df_symbolic(x):
    return np.array(df_num_raw(x[0], x[1]))

test_point = np.array([0.2, 0.4])
print(f"Manual df(x):   {df(test_point)}")
print(f"Symbolic df(x): {df_symbolic(test_point)}")

In [ ]:
# Now we can run the optimization using the symbolic gradient
trace_sym = grad_descent(df_symbolic, [2, -0.3])

print(f"Final point found: {trace_sym[-1]['x']}")
print(f"Iterations: {len(trace_sym)}")

Similar to numerical differentation, symbolic differentiation becomes impractical for models with billions of parameters because of "expression swell," a phenomenon where the algebraic formula for the gradient grows exponentially in size as it repeatedly applies the chain rule across millions of nested operations. This results in massive, unwieldy expressions that can exhaust a computer's memory and take far longer to evaluate than the original function itself.

## Automatic differentiation

Finally, we come to **Automatic differentiation (AD)**. This is the engine that powers libraries like Jax, PyTorch, and TensorFlow.

Unlike numerical differentiation, it doesn't approximate. Unlike symbolic differentiation, it doesn't try to build one giant algebraic formula. Instead, it breaks the function down into a **computational graph** of basic operations (addition, multiplication, etc.). 

Each "node" in the graph knows how to compute its own derivative locally. To find the gradient of the whole function, we just apply the **chain rule** step-by-step, moving backward through the graph from the output to the inputs. This is known as **backpropagation**.

**Why it wins:**
- **Exactness:** No $\delta$ or rounding errors.
- **Efficiency:** We can compute the gradient of *billions* of parameters with just one "forward pass" and one "backward pass."

Note that the code below mostly comes from `micrograd`, written by Andrej Karpathy, which he uses to illustrate how automatic differentiation works in the context of neural networks. We added the `exp` function so that we could execute automatic differentiation on our example which includes an `exp`.

You can find his repository [here](https://github.com/karpathy/micrograd/tree/master)

In [ ]:
import math


class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op 

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')
        
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def backward(self):
        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

Now, let's use this `Value` class to calculate the gradient of our function $f(x) = -\exp^{-(x_0^2 + x_1^2)}$ at the same test point we used before.

In [ ]:
# 1. Define inputs as 'Value' objects
x0 = Value(0.2)
x1 = Value(0.4)

# 2. Define the function (Notice how it looks like normal math!)
# f = -exp(-(x0^2 + x1^2))
f = -((x0**2 + x1**2) * -1).exp()

# 3. Trigger backpropagation
f.backward()

print(f"Function output: {f.data}")
print(f"Autograd df/dx: {np.array([x0.grad, x1.grad])}")

# Comparison with our symbolic/manual result
print(f"\nPrevious Symbolic Result: {df_symbolic([0.2, 0.4])}")

In [ ]:
def df_autograd(x):
    """
    A wrapper that takes a numpy array x and returns 
    the gradient calculated via automatic differentiation.
    """
    # 1. Wrap inputs in Value objects
    x0 = Value(x[0])
    x1 = Value(x[1])
    
    # 2. Compute the function f(x) = -exp(-(x0^2 + x1^2))
    # Note: we use our .exp() method defined in the Value class
    f = -((x0**2 + x1**2) * -1).exp()
    
    # 3. Compute gradients via backpropagation
    f.backward()
    
    # 4. Return as a numpy array for the optimizer
    return np.array([x0.grad, x1.grad])

# Run the gradient descent using the Autograd engine
trace_autograd = grad_descent(df_autograd, [2.0, -0.3])

# Results
print(f"Optimization finished in {len(trace_autograd)} iterations.")
print(f"Final point: {trace_autograd[-1]['x']}")

### How many times do we evaluate the function?

One might assume that if a model has 1 billion parameters, we would need to run the function 1 billion times to see how each parameter affects the output. This is true for **Numerical Differentiation**.

However, with **Automatic Differentiation (Reverse-Mode)**, the cost is significantly lower:

1.  **The Forward Pass:** We evaluate the function once to get the result. During this pass, the computer "remembers" the intermediate steps and values.
2.  **The Backward Pass:** We traverse the computational graph in reverse to calculate the gradients. 

**The result:** Computing the gradient for every single parameter (no matter if there are 2 or 2 billion) costs roughly **2 to 5 times** the computational effort of a single forward pass. 

### Summary of Methods

We have now calculated the gradient for the same function using three distinct methods:

1.  **Numerical ($\delta$):** Easy to implement but computationally expensive for high dimensions and prone to approximation errors.
2.  **Symbolic (`sympy`):** Exact, but leads to "expression swell" where formulas become too large to handle for complex models.
3.  **Automatic (Autograd):** The gold standard for Deep Learning. It is exact, efficient, and allows us to calculate gradients for models with billions of parameters by simply tracking the "recipe" of the calculation.